# Synax Code — train model Python từ số 0

Dataset: `codeparrot/codeparrot-clean`, chỉ giữ license permissive
(Apache-2.0, MIT, BSD, ISC, Unlicense, CC0). Xem `docs/02-python.md`.

Notebook chạy thử 200MB trước, sau đó mới chạy full 4GB.


In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

REPO = "/content/Synax"
if not os.path.exists(REPO):
    !git clone https://github.com/wiothemilo-lang/Synax.git {REPO}
os.chdir(REPO)
!pip install -q -e ".[data]"


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/synax"
os.makedirs(DRIVE, exist_ok=True)
print("drive:", DRIVE)


## 1. Chuẩn bị corpus Python

Cell dưới stream dataset, lọc license + AST + dedupe, ghi JSONL vào Drive.

- Chạy thử: `MAX_BYTES = 200_000_000` (~200MB).
- Full: `MAX_BYTES = 4_000_000_000` (~4GB, khoảng 1B token).

Nếu đã có file, cell sẽ bỏ qua.


In [ ]:
from pathlib import Path

DATA = f"{DRIVE}/python"
CORPUS = f"{DATA}/clean/python.jsonl"
Path(CORPUS).parent.mkdir(parents=True, exist_ok=True)

MAX_BYTES = 200_000_000  # đổi thành 4_000_000_000 cho full
if not Path(CORPUS).exists():
    !python -m synax.data.code --output {CORPUS} --max-bytes {MAX_BYTES} --progress-every 20000
else:
    print("corpus exists:", CORPUS)


In [ ]:
TOKENIZER = f"{DRIVE}/tokenizer/sp32-code"
Path(TOKENIZER).parent.mkdir(parents=True, exist_ok=True)

if not Path(TOKENIZER + ".model").exists():
    !python -m synax.data.tokenizer train --input {CORPUS} --output-prefix {TOKENIZER} --vocab-size 32000 --code --tmp-dir {DATA}
else:
    print("tokenizer exists:", TOKENIZER + ".model")


In [ ]:
TRAIN_BIN = f"{DATA}/tokenized/train.bin"
VAL_BIN = f"{DATA}/tokenized/val.bin"
Path(TRAIN_BIN).parent.mkdir(parents=True, exist_ok=True)

if not Path(TRAIN_BIN).exists():
    !python -m synax.data.tokenizer encode --input {CORPUS} --tokenizer {TOKENIZER}.model --output {TRAIN_BIN} --val-output {VAL_BIN} --val-ratio 0.005 --code
else:
    print("shards exist")


## 2. Chạy thử nhanh (200 step)

Kiểm tra model + checkpoint + sample trên corpus 200MB.


In [ ]:
OUT_TEST = f"{DRIVE}/checkpoints/60m-code-test"
Path(OUT_TEST).mkdir(parents=True, exist_ok=True)

!python -m synax.train.pretrain \
  --config configs/model-60m-code.yaml \
  --train-bin {TRAIN_BIN} \
  --val-bin {VAL_BIN} \
  --tokenizer {TOKENIZER}.model \
  --out-dir {OUT_TEST} \
  --steps 200


## 3. Chạy đầy đủ

Chỉ chạy khi đã có ~1B token (~4GB text). Session ngắt thì chạy lại với
`--resume auto` là tiếp tục.


In [ ]:
OUT = f"{DRIVE}/checkpoints/60m-code"
Path(OUT).mkdir(parents=True, exist_ok=True)

!python -m synax.train.pretrain \
  --config configs/model-60m-code.yaml \
  --train-bin {TRAIN_BIN} \
  --val-bin {VAL_BIN} \
  --tokenizer {TOKENIZER}.model \
  --out-dir {OUT} \
  --resume auto


In [ ]:
from pathlib import Path

RUN_DIR = Path(f"{DRIVE}/checkpoints/60m-code")
if not (RUN_DIR / "latest.txt").exists():
    RUN_DIR = Path(f"{DRIVE}/checkpoints/60m-code-test")
latest = (RUN_DIR / "latest.txt").read_text().strip()
print("checkpoint:", RUN_DIR / latest)

!python -m synax.inference.generate \
  --checkpoint {RUN_DIR}/{latest} \
  --tokenizer {TOKENIZER}.model \
  --prompt "def fibonacci(n):" \
  --max-new-tokens 120


## Ghi chú

- 200MB chỉ để test pipeline; model thật cần ~4GB text (~1B token).
- Checkpoint giữ 2 bản mới nhất trong Drive.
- Không commit data/checkpoint vào git.
- Nếu muốn tăng tốc tải dataset, đặt `HF_TOKEN` trong Colab secrets.
